# arxiv-rag: Colab quickstart (GPU)
Builds the corpus, indexes it on the GPU, generates a test set and records the baseline score.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then add your LLM key under the 🔑 *Secrets* panel on the left as `GEMINI_API_KEY` (free key: https://aistudio.google.com/apikey) and turn on *Notebook access*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
# Safe to re-run: clones the first time, pulls the latest fixes after that
!git clone -q https://github.com/Sagar00789/arxiv-rag.git /content/arxiv-rag 2>/dev/null || git -C /content/arxiv-rag pull -q
%cd /content/arxiv-rag
!pip install -q -e ".[all]"

## 2. API key (read from Colab Secrets, never typed into the notebook)

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["LLM_MODEL"] = "gemini/gemini-3.5-flash-lite"  # any LiteLLM model string works

In [ ]:
!arxiv-rag check-llm  # fails in seconds if the model name or key is wrong

### Coming back? Restore last session's corpus and index from Google Drive
If you saved to Drive before (last cell), run this and **skip steps 3–4**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/arxiv-rag/data /content/drive/MyDrive/arxiv-rag/qdrant_data . && ls data

## 3. Download and chunk ~50 papers on RAG and LLM agents (≈3–5 min)

In [ ]:
!arxiv-rag ingest --max-results 50

## 4. Embed and index (uses the GPU automatically)

In [ ]:
!arxiv-rag index

## 5. Try it

In [ ]:
!arxiv-rag ask "How does Self-RAG decide when to retrieve?"

## 6. Build the test set
The LLM writes one question per sampled chunk; that chunk becomes the gold answer.
It's paced for the free tier (~4 min for 50) and saves as it goes: **if it stops on a quota limit, just re-run the cell and it resumes.**
**Skim the questions below and delete any that are vague or unanswerable.** A clean test set is what makes the numbers credible.

In [ ]:
!arxiv-rag make-testset --n 50

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.read_json("eval/testset.jsonl", lines=True)[["question", "gold_paper_id", "reference_answer"]].head(15)

## 7. Compare retrievers
- **dense**: embeddings only (the baseline)
- **bm25**: keyword search only
- **hybrid**: dense + BM25 merged with reciprocal rank fusion
- **hybrid-rerank**: hybrid's top 30 re-scored by a cross-encoder (`bge-reranker-base`, uses the GPU)

Each one adds a row to `eval/RESULTS.md`.

In [ ]:
!arxiv-rag eval -r dense -r bm25 -r hybrid -r hybrid-rerank
!cat eval/RESULTS.md

## 8. Save your work
Downloads the results. Upload them back to Claude so they go into the repo.

In [ ]:
from google.colab import files
!zip -q -r eval_bundle.zip eval/RESULTS.md eval/results
files.download("eval_bundle.zip")

Optional: keep the corpus and index in Google Drive so the next session skips steps 3–4.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/arxiv-rag && cp -r data qdrant_data /content/drive/MyDrive/arxiv-rag/